In [3]:
import numpy as np

data = np.load('octmnist.npz')
print("Keys in octmnist.npz: ", data.files)

for k in data.files:
    arr = data[k]
    print(f"{k:>12}: shape = {arr.shape}")

train_images = data["train_images"]
train_labels = data["train_labels"].ravel()

val_images = data["val_images"]
val_labels = data["val_labels"].ravel()

test_images = data["test_images"]
test_labels = data["test_labels"].ravel()

Keys in octmnist.npz:  ['train_images', 'val_images', 'test_images', 'train_labels', 'val_labels', 'test_labels']
train_images: shape = (97477, 28, 28)
  val_images: shape = (10832, 28, 28)
 test_images: shape = (1000, 28, 28)
train_labels: shape = (97477, 1)
  val_labels: shape = (10832, 1)
 test_labels: shape = (1000, 1)


Loading Necessary Data. Taken Straight out of a Previous Notebook

In [4]:
from models import stratified_subset
from sklearn.utils.class_weight import compute_sample_weight
import tensorflow as tf
from pathlib import Path
from time import strftime

x_tr, y_tr = stratified_subset(train_images, train_labels, 22000)
x_va, y_va = stratified_subset(val_images, val_labels, 2000)

sample_weights = compute_sample_weight(class_weight='balanced', y=y_tr)

early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=10,
                                                     restore_best_weights=True)

def get_run_logdir(root_logdir='my_logs'):
    return Path(root_logdir)/strftime("run_%Y_%m_%d_%H_%M_%S")

run_logdir = get_run_logdir()

tensorboard_cb = tf.keras.callbacks.TensorBoard(run_logdir,
                                                profile_batch=(100, 200))


Early Prep. Also Taken Straight out a Previous Notebook.

In [6]:
x_tr_flat = x_tr.reshape(x_tr.shape[0], -1)

### Model Tuning

In [4]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV
from models import median, avg_2x2_pool, flatten_data

x_tr_stacking = flatten_data(avg_2x2_pool(median(x_tr))).astype(np.float32)

Model 1: Preprocessed Stacking

In [5]:
from models import build_preprocessed_stacking, build_preprocessed_stacking_tunable

stacking_param_dist = {
    "svc__kernel__n_components": [300, 500, 700],
    "svc__svc__C":               loguniform(0.1, 10),
    "final_estimator__C":        loguniform(0.1, 10),
}

stacking_search = RandomizedSearchCV(
    estimator=build_preprocessed_stacking_tunable(),
    param_distributions=stacking_param_dist,
    n_iter=10,
    cv=2,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
    random_state=42,
    refit=True,
)

stacking_search.fit(x_tr_stacking, y_tr)

print("Best stacking params:", stacking_search.best_params_)
print(f"Best CV accuracy: {stacking_search.best_score_:.4f}")
stacking_clf_tuned = build_preprocessed_stacking()
stacking_clf_tuned.set_params(**stacking_search.best_params_)

Fitting 2 folds for each of 10 candidates, totalling 20 fits


[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:   20.5s finished
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:   10.2s finished


Best stacking params: {'final_estimator__C': np.float64(8.706020878304859), 'svc__kernel__n_components': 500, 'svc__svc__C': np.float64(0.26587543983272705)}
Best CV accuracy: 0.7059


,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('rf', ...), ('svc', ...)]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegre...max_iter=1000)
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",2
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",2
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function med...002A3AB651E40>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion 

Note: n_iter and cv were reduced to decrease training speed. Likely at the cost of performance.

Model 2: Soft Voting Classifier

In [ ]:
from models import build_voting_classifier

voting_param_dist = {
    "rf__model__n_estimators":     randint(100, 400),
    "rf__model__max_features":     [0.2, 0.25, 0.33, "sqrt"],
    "rf__model__min_samples_leaf": randint(1, 8),
    "rf__model__max_depth":        [None, 20, 40],
    "ada__model__n_estimators":    randint(100, 400),
    "ada__model__learning_rate":   loguniform(0.5, 2.0),
}

voting_search = RandomizedSearchCV(
    estimator=build_voting_classifier(),
    param_distributions=voting_param_dist,
    n_iter=8,
    cv=2,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
    random_state=42,
    refit=True,
)

voting_search.fit(x_tr, y_tr)

print("Best voting params:", voting_search.best_params_)
print(f"Best CV accuracy: {voting_search.best_score_:.4f}")
voting_clf_tuned = voting_search.best_estimator_

Fitting 2 folds for each of 8 candidates, totalling 16 fits


Model 3: SVD Soft Voting

In [ ]:
from models import build_svd_soft_voting

svd_param_dist = {
    "svd__n_components":   [75, 100, 150, 200],
    "model__sgd__alpha":   loguniform(1e-5, 1e-2),
    "model__sgd__penalty": ["l2", "elasticnet"],
    "model__svc__C":       loguniform(0.1, 10),
    "model__svc__gamma":   ["scale", "auto"],
}

svd_search = RandomizedSearchCV(
    estimator=build_svd_soft_voting(),
    param_distributions=svd_param_dist,
    n_iter=10,
    cv=2,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
    random_state=42,
    refit=True,
)

svd_search.fit(x_tr, y_tr)

print("Best SVD soft-voting params:", svd_search.best_params_)
print(f"Best CV accuracy: {svd_search.best_score_:.4f}")
svd_clf_tuned = svd_search.best_estimator_

Model 4: Wide & Deep

In [7]:
from functools import partial
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from models import build_tuned_wide_deep

x_va_flat = x_va.reshape(x_va.shape[0], -1).astype(np.float32)

scaler_wd = MinMaxScaler()
x_tr_s = scaler_wd.fit_transform(x_tr_flat)
x_va_s = scaler_wd.transform(x_va_flat)

thresholds = ["0.90", "0.95", "0.99", "0.999"]
n_by_threshold = {}
for t in thresholds:
    p = PCA(n_components=float(t), random_state=42).fit(x_tr_s)
    n_by_threshold[t] = p.n_components_

n_max = n_by_threshold["0.999"]

pca_full = PCA(n_components=n_max, random_state=42).fit(x_tr_s)
x_tr_pca = pca_full.transform(x_tr_s).astype(np.float32)
x_va_pca = pca_full.transform(x_va_s).astype(np.float32)

norm_raw = tf.keras.layers.Normalization()
norm_raw.adapt(x_tr_flat.astype(np.float32))
mean, var, _ = norm_raw.get_weights()

x_tr_wd = np.concatenate([x_tr_pca, x_tr_flat.astype(np.float32)], axis=1)
x_va_wd = np.concatenate([x_va_pca, x_va_flat.astype(np.float32)], axis=1)


np.int64(0)

To explain: The way the wide and deep network was implemented was as a KerasClassifier shown to us in the TextBook. Because of this PCA was implemented as part of a pipeline as you can see in models.py. This became challenging once it came time to tune though. I wanted the flexibility of architecture screening that Keras Tuning provided but in the current implementation of Wide And Deep that was impossible. 

Above is the solution. Basically all Preprocessing steps are done ahead of time.
PCA thresholds are calculated ahead of time. 
Then for the maximum number of components PCA is performed

Normalization mean and variance are extracted ahead of time as well.

The threshold dictionary, maximum number of PCA components, mean, and variance are then all passed to the model at build time

depepending on the threshold the tuner selects only that number of PCA components is fed to the Wide Network.
All of the regular inputs are always fed to the deep network. This is why we included n_max so we always know what that cutoff is.
Finally the mean and variancce are used in the normalization layer of the model.

For actual code implementation please see the attached models.py file

finally a "partial function" is used so we can actually pass these hyperparameters to the tuner.

This was taken from this stack overflow: https://stackoverflow.com/questions/69790356/how-to-pass-fix-hyperparameters-as-variables-for-keras-tuner

In [ ]:
import keras_tuner as kt

wd_tuner = kt.Hyperband(
    hypermodel=partial(
        build_tuned_wide_deep,
        n_by_threshold=n_by_threshold,
        n_max=n_max,
        mean=mean,
        variance=var,
    ),
    objective="val_accuracy",
    max_epochs=20,
    factor=3,
    hyperband_iterations=1,
    directory="./wd_tuning",
    project_name="wd_search",
    overwrite=True,
    seed=42,
)

wd_tuner.search(
    x_tr_wd, y_tr,
    validation_data=(x_va_wd, y_va),
    sample_weight=sample_weights,
    batch_size=128,
    callbacks=[early_stopping_cb],
    verbose=1,
)

Batch size of 128 was selected for speed.

In [ ]:
best_wd_hp = wd_tuner.get_best_hyperparameters(num_trials=1)[0]
best_wd_model = wd_tuner.hypermodel.build(best_wd_hp)
best_wd_model.summary()


In [ ]:
from models import preprocess_for_cnn

x_tr_cnn = preprocess_for_cnn(x_tr)
x_va_cnn = preprocess_for_cnn(x_va)

Preprocess for CNN is a simple wrapper function that both median filters and adds the necessary channel

Model 5: CNN

In [ ]:
import keras_tuner as kt
from models import build_tuned_cnn

cnn_tuner = kt.Hyperband(
    hypermodel=build_tuned_cnn,
    objective="val_accuracy",
    max_epochs=20,
    factor=3,
    hyperband_iterations=1,
    directory="./cnn_tuning",
    project_name="cnn_search",
    overwrite=True,
    seed=42,
)

cnn_tuner.search(
    x_tr_cnn, y_tr,
    validation_data=(x_va_cnn, y_va),
    sample_weight=sample_weights,
    batch_size=128,
    callbacks=[early_stopping_cb],
    verbose=1,
)

In [ ]:
best_cnn_hp  = cnn_tuner.get_best_hyperparameters(num_trials=1)[0]
best_cnn_model = cnn_tuner.hypermodel.build(best_cnn_hp)
best_cnn_model.summary()

ResNet — Hyperband Tuning

In [ ]:
from models import prep_for_cnn

x_tr_res = prep_for_cnn(x_tr)
x_va_res = prep_for_cnn(x_va)

No median Filtering done here

In [ ]:
from models import build_tuned_resnet

resnet_tuner = kt.Hyperband(
    hypermodel=build_tuned_resnet,
    objective="val_accuracy",
    max_epochs=20,
    factor=3,
    hyperband_iterations=1,
    directory="./resnet_tuning",
    project_name="resnet_search",
    overwrite=True,
    seed=42,
)

resnet_tuner.search(
    x_tr_res, y_tr,
    validation_data=(x_va_res, y_va),
    sample_weight=sample_weights,
    batch_size=128,
    callbacks=[early_stopping_cb],
    verbose=1,
)

In [ ]:
best_resnet_hp    = resnet_tuner.get_best_hyperparameters(num_trials=1)[0]
best_resnet_model = resnet_tuner.hypermodel.build(best_resnet_hp)
best_resnet_model.summary()

## Final Evaluation on Test Set

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, f1_score, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_sample_weight

class_names = ["Normal", "CNV", "DME", "Drusen"]

sample_weights_full = compute_sample_weight(class_weight="balanced", y=train_labels)

all_results = []

def evaluate_classifier(name, y_true, y_pred, y_proba, time):
    """
    Evalutes model for us, plots a confusion matrix, and ROC curve for each
    """
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=1)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=1)

    print(f"{name}")
    print(f"Accuracy:           {acc:.4f}")
    print(f"Weighted Precision: {prec:.4f}")
    print(f"Weighted F1:        {f1:.4f}")
    print(f"Training Time:      {time:.4f}s")

    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        display_labels=class_names,
        ax=ax[0],
        colorbar=False,
        cmap="plasma",
    )
    ax[0].set_title(f"{name} — Confusion Matrix")
    ax[0].tick_params(axis="x", rotation=15)

    y_bin  = label_binarize(y_true, classes=[0, 1, 2, 3])
    colors = ["r", "b", "g", "y"]
    aucs   = []
    for i, (cname, col) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        ax[1].plot(fpr, tpr, color=col, lw=1.8,
                   label=f"{cname} (AUC={roc_auc:.3f})")
    ax[1].plot([0, 1], [0, 1], "k--", lw=1)
    ax[1].set_xlabel("False Positive Rate")
    ax[1].set_ylabel("True Positive Rate")
    ax[1].set_title(f"{name} — ROC Curves (OvR)")
    ax[1].legend(loc="lower right")

    mean_auc = np.mean(aucs)

    plt.tight_layout()
    plt.show()
    return {"name": name, "acc": acc, "prec": prec, "f1": f1, "time": time, "auc": mean_auc}


To quickly explain what the above function does (it's fairly straight forward though). 

After the model is trained the accurary (weighted recall), weighted precision and f1 score are all calculated and printed.
A confusion matrix is the calculated and displayed
then does the same with AUC curves per class
finally it calculates the Mean AUC and returns a dictionary of all results 
including training time (passed along to function ahead of time)



Model 1: Preprocessed Stacking

In [ ]:
start = time.time()
stacking_clf_tuned.fit(train_images, train_labels)
end = time.time()

t = end - start
stacking_pred  = stacking_clf_tuned.predict(test_images)
stacking_proba = stacking_clf_tuned.predict_proba(test_images)
all_results.append(
    evaluate_classifier("Preprocessed Stacking", test_labels, stacking_pred, stacking_proba, time=t)
)

Model 2: Soft Voting Classifier

In [ ]:
start = time.time()
voting_clf_tuned.fit(train_images, train_labels)
end = time.time()

t = end - start
voting_pred  = voting_clf_tuned.predict(test_images)
voting_proba = voting_clf_tuned.predict_proba(test_images)
all_results.append(
    evaluate_classifier("Soft Voting", test_labels, voting_pred, voting_proba, time=t)
)

Model 3: SVD Soft Voting

In [ ]:
start = time.time()
svd_clf_tuned.fit(train_images, train_labels)
end = time.time()

t = end - start
svd_pred  = svd_clf_tuned.predict(test_images)
svd_proba = svd_clf_tuned.predict_proba(test_images)
all_results.append(
    evaluate_classifier("SVD Soft Voting", test_labels, svd_pred, svd_proba, time=t)
)

Model 4: Wide & Deep

In [ ]:
x_tr_flat_full = train_images.reshape(len(train_images), -1).astype(np.float32)
x_tr_wd_full = np.concatenate([
    pca_full.transform(scaler_wd.transform(x_tr_flat_full)).astype(np.float32),
    x_tr_flat_full,
], axis=1)

x_te_flat = test_images.reshape(len(test_images), -1).astype(np.float32)
x_te_wd = np.concatenate([
    pca_full.transform(scaler_wd.transform(x_te_flat)).astype(np.float32),
    x_te_flat,
], axis=1)

wd_final = wd_tuner.hypermodel.build(best_wd_hp)
start = time.time()
wd_final.fit(
    x_tr_wd_full, train_labels,
    epochs=100,
    batch_size=128,
    sample_weight=sample_weights_full,
    validation_split=0.1,
    callbacks=[early_stopping_cb],
    verbose=1,
)
end = time.time()

t = end - start
wd_proba = wd_final.predict(x_te_wd)
wd_pred  = np.argmax(wd_proba, axis=1)
all_results.append(
    evaluate_classifier("Wide & Deep", test_labels, wd_pred, wd_proba, time=t)
)

Prepare Full-Dataset CNN Inputs

In [ ]:
x_tr_cnn_full = preprocess_for_cnn(train_images)
x_te_cnn      = preprocess_for_cnn(test_images)

Model 5: CNN

In [ ]:
cnn_final = build_tuned_cnn(best_cnn_hp, n_train_samples=len(train_images))
start = time.time()
cnn_final.fit(
    x_tr_cnn_full, train_labels,
    epochs=100,
    batch_size=128,
    sample_weight=sample_weights_full,
    validation_split=0.1,
    callbacks=[early_stopping_cb],
    verbose=1,
)
end = time.time()

t = end - start
cnn_proba = cnn_final.predict(x_te_cnn)
cnn_pred  = np.argmax(cnn_proba, axis=1)
all_results.append(
    evaluate_classifier("CNN", test_labels, cnn_pred, cnn_proba, time=t)
)

Model 6: Resnet

In [ ]:
x_tr_res_full = prep_for_cnn(train_images)
x_te_res_full = prep_for_cnn(test_images)

In [ ]:
resnet_final = resnet_tuner.hypermodel.build(best_resnet_hp)
start = time.time()
resnet_final.fit(
    x_tr_res_full, train_labels,
    epochs=100,
    batch_size=128,
    sample_weight=sample_weights_full,
    validation_split=0.1,
    callbacks=[early_stopping_cb],
    verbose=1,
)
end = time.time()

t = end - start
resnet_proba = resnet_final.predict(x_te_res_full)
resnet_pred  = np.argmax(resnet_proba, axis=1)
all_results.append(
    evaluate_classifier("ResNet", test_labels, resnet_pred, resnet_proba, time=t)
)

### Final Summary

In [ ]:
col_names = {
    "acc":  "Accuracy",
    "prec": "Weighted Precision",
    "f1":   "Weighted F1",
    "time": "Train Time (s)",
    "auc":  "Mean AUC",
}

summary_df = (
    pd.DataFrame(all_results)
    .set_index("name")
    .rename(columns=col_names)
    .sort_values("Weighted F1", ascending=False)
)

print(summary_df.to_string(float_format="{:.4f}".format))